In [ ]:
import pandas as pd
import numpy as np
import ast

import os
import matplotlib.pyplot as plt
import seaborn as sns
from plinder_analysis_utils import DockingAnalysisBase, PoseBustersAnalysis, PropertyAnalysis

import statsmodels.formula.api as smf


In [ ]:
PLINDER_TEST_COLUMNS = [
    "system_id", "ligand_smiles",
    # binary 
    # "ligand_is_covalent", "ligand_is_ion", "ligand_is_cofactor", "ligand_is_artifact",
    # discrete
    "system_num_protein_chains",
    "ligand_num_rot_bonds", "ligand_num_hbd", "ligand_num_hba", "ligand_num_rings",
    # continuous
    "entry_resolution", "entry_validation_molprobity", 
    "system_num_pocket_residues", "system_num_interactions",
    "ligand_molecular_weight", "ligand_crippen_clogp", 
    "ligand_num_interacting_residues", "ligand_num_neighboring_residues", "ligand_num_interactions",
]
# Create category mapping for visualization
CATEGORY_MAPPING = {
    "ligand_is_covalent": "binary",
    "ligand_is_ion": "binary",
    "ligand_is_cofactor": "binary",
    "ligand_is_artifact": "binary",
    "system_num_protein_chains": "discrete",
    "ligand_num_rot_bonds": "continuous",    
    "ligand_num_hbd": "continuous",
    "ligand_num_hba": "continuous",
    "ligand_num_rings": "continuous",
    "entry_resolution": "continuous",
    "entry_validation_molprobity": "continuous",
    "system_num_pocket_residues": "continuous",
    "system_num_interactions": "continuous",
    "ligand_molecular_weight": "continuous",
    "ligand_crippen_clogp": "continuous",
    "ligand_num_interacting_residues": "continuous",
    "ligand_num_neighboring_residues": "continuous",
    "ligand_num_interactions": "continuous",
    "ligand_is_artifact": "binary"     
}

In [ ]:
df_combined = pd.read_csv("plinder_set_0_annotated.csv")

In [ ]:
# build a boolean mask: drop any row where covalent, ionic or has_ion is True
mask = ~(
    df_combined['ligand_is_covalent'] |
    df_combined['ligand_is_ion'] |
    df_combined['has_ion'] |
    df_combined['ligand_is_cofactor']
)

# filter and reset index
df_combined = df_combined.loc[mask].reset_index(drop=True)
print("Filtered shape:", df_combined.shape)

In [ ]:
# First analyze multiple properties
property_analysis = PropertyAnalysis(df_combined)
methods = ["surfdock", "gnina", "chai-1", "diffdock_pocket_only", "icm", "vina"]

## Complementarity Between Physics-based and ML-based: Mixed Effect Analysis

### Prepare the df

In [ ]:
MIXED_EFFECT_VARS = [
    "protein", "rmsd","method",
    # "system_id", "ligand_smiles",
    # binary 
    # "ligand_is_covalent", "ligand_is_ion", "ligand_is_cofactor", "ligand_is_artifact",
    # discrete
    # "system_num_protein_chains",
    "ligand_num_rot_bonds", "ligand_num_hbd", "ligand_num_hba", "ligand_num_rings",
    # continuous
    "entry_resolution", "entry_validation_molprobity", 
    # "system_num_pocket_residues", 
    "system_num_interactions",
    "ligand_molecular_weight", "ligand_crippen_clogp", 
    "ligand_num_interacting_residues", 
    "ligand_num_neighboring_residues", 
    # "ligand_num_interactions",
]

In [ ]:
df_mixed = df_combined[MIXED_EFFECT_VARS]
# Create a Method_Type column based on the classification
df_mixed['Method_Type'] = df_mixed['method'].apply(
    lambda x: 'ML' if x in ['chai-1', 'diffdock_pocket_only', 'surfdock'] else 'Physics'
)

# Display the counts of each method type
print(df_mixed['Method_Type'].value_counts())

# Verify the classification
for method in df_mixed['method'].unique():
    method_type = 'ML' if method in ['chai-1', 'diffdock_pocket_only', 'surfdock'] else 'Physics'
    print(f"{method}: {method_type}")

### system_num_protein_chains

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="system_num_protein_chains",
    bins=[1,2,3,4]
)
print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="system_num_protein_chains",
    bins=[1,2,3,4]
)
print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="system_num_protein_chains",
    bins=[1,2,3,4]
)
print("Stratified McNemar p =", stratified_analysis['cmh'])

### ligand_num_rot_bonds

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rot_bonds",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rot_bonds",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rot_bonds",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### ligand_num_hbd

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hbd",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hbd",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hbd",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

#### ligand_num_hba

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hba",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hba",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_hba",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### ligand_num_rings

In [ ]:
df_combined['ligand_num_rings'].value_counts()

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rings",
    bins=[0,3,6,9,12]
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rings",
    bins=[0,3,6,9,12]
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_rings",
    bins=[0,3,6,9,12]
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### entry_resolution

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="entry_resolution",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="entry_resolution",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="entry_resolution",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### entry_validation_molprobity

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="entry_validation_molprobity",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="entry_validation_molprobity",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="entry_validation_molprobity",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### system_pocket_residues

### system_num_interactions

### ligand_molecular_weight

### ligand_crippen_clogp

### ligand_num_interactions

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="system_num_interactions",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="system_num_interactions",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="system_num_interactions",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

### ligand_num_neighboring_residues

### ligand_num_interacting_residues

In [ ]:
# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only"],
    methods_group2=["icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_interacting_residues",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only","surfdock", "chai-1"],
    methods_group2=["vina", "gnina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_interacting_residues",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

# stratified_analysis = property_analysis.stratified_success_analysis(
stratified_analysis = property_analysis.stratified_mcnemar_analysis(
    methods_group1=["diffdock_pocket_only", "chai-1", "surfdock"],
    methods_group2=["vina", "icm"],
    rmsd_threshold=2.0,
    property_name="ligand_num_interacting_residues",
)

print("Stratified McNemar p =", stratified_analysis['cmh'])

## Distribution of the features

In [ ]:
properties = [
    "ligand_num_rot_bonds",
    "ligand_num_hbd",
    "ligand_num_hba",
    "ligand_num_rings",
    "entry_resolution",
    "entry_validation_molprobity",
    "syestem_num_interactions",
    "ligand_molecular_weight",
    "ligand_crippen_clogp",
    "ligand_num_interacting_residues",
    "ligand_num_neighboring_residues",
]

In [ ]:
# First run the complementary analysis
property_analysis = PropertyAnalysis(df_combined)
complementary_results = property_analysis.complementary_success_analysis(
    method1=['diffdock_pocket_only', 'chai-1', 'surfdock'],  # ML methods
    method2=['vina', 'icm'],                   # Physics-based methods
    rmsd_threshold=2.0,
    plot=True
)

# Then analyze property distributions across complementary cases
property_distributions = property_analysis.compare_property_distributions_in_complementary_cases(
    complementary_results=complementary_results,
    properties=properties,
    method_labels=('ML-based', 'Physics-based'),
    property_types={
        'ligand_molecular_weight': 'continuous',
        'ligand_num_rot_bonds': 'continuous',
        'ligand_num_hbd': 'continuous',
        'ligand_num_hba': 'continuous',
        'ligand_num_rings': 'continuous',
        'entry_resolution': 'continuous',
        'entry_validation_molprobity': 'continuous',
        'system_num_interactions': 'continuous',
        'ligand_crippen_clogp': 'continuous',
        'ligand_num_interacting_residues': 'continuous',
        'ligand_num_neighboring_residues': 'continuous'
    },
    plot=True
)

# To view the summary statistics
print(property_distributions['summary_stats']['ligand_molecular_weight'])

# To access the statistical test results
print(property_distributions['test_results']['ligand_num_rot_bonds'])

# Follow-up analysis

### correlation between ligand_num_hbd and ligand_crippen_clogp

In [ ]:
from scipy.stats import pearsonr
r, p = pearsonr(df_combined['ligand_num_hbd'].fillna(0), df_combined['ligand_crippen_clogp'].fillna(0)) 
print(f"r = {r:.2f}, p = {p:.1e}")

### Ring-count × Rotatable-bond interaction
	•	Ring count and rotatable bonds (RBs) capture different flavours of flexibility:
	•	Too few rings → high RBs → entropic penalty & sampling explosion.
	•	Too many fused rings → RBs “frozen” but ligand may become too planar/rigid to explore alternate poses.
	•	The difficult corners lie where the two variables disagree:
	•	Low rings / high RBs = floppy aliphatic chains.
	•	High rings / low RBs = flat polycycles (PAHs, macrocycles).

In [ ]:
# 2-D heat map of success rate
df_combined['success'] = df_combined['rmsd_≤_2å']
heat = (
    df_combined.assign(
        ring_bin = pd.cut(df_combined.ligand_num_rings, bins=[0,1,3,5,99], labels=['0-1','2-3','4-5','>5']),
        rb_bin   = pd.cut(df_combined.ligand_num_rot_bonds, bins=[0,2,5,8,99], labels=['0-2','3-5','6-8','>8'])
    )
    .groupby(['ring_bin','rb_bin'])['success']
    .mean()
    .unstack()
)
sns.heatmap(heat*100, annot=True, fmt='.0f', cmap='viridis'); plt.title('Success % vs rings × RBs');

### mixed effect analysis

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
# Define variables with standardized naming for consistency
df_combined['ring_c'] = df_combined.ligand_num_rings
df_combined['rb_c'] = df_combined.ligand_num_rot_bonds

# Ensure success is properly defined as a binary integer
df_combined['success'] = (df_combined['rmsd'] <= 2.0).astype(int)

# Use mixed effects logistic regression with an interaction term
m_int = smf.mixedlm(
    'rmsd ~ ring_c * rb_c + C(method)',
    data=df_combined, 
    groups=df_combined['protein'],
    family=sm.families.Binomial()  # This is key for binary outcomes
).fit()
print(m_int.summary())

In [ ]:
sns.scatterplot(
    x='ligand_num_rings', y='ligand_num_rot_bonds',
    hue='ligand_num_rings', style='success', data=df_combined, palette='viridis'
)

### molecular weight

In [ ]:
# Fraction of successes by MW band
best_rmsd_df = df_combined.groupby('protein').apply(
    lambda x: x.loc[x['rmsd'].idxmin()] if not x['rmsd'].isna().all() else x.iloc[0]
).reset_index(drop=True)

best_rmsd_df['mw_bin'] = pd.cut(best_rmsd_df.ligand_molecular_weight, bins=[0,300,400,500,800])
success_rate = best_rmsd_df.groupby('mw_bin')['success'].mean()*100
print(success_rate)

df_combined['mw_bin'] = pd.cut(df_combined.ligand_molecular_weight, bins=[0,300,400,500,800])
success_rate = df_combined.groupby('mw_bin')['success'].mean()*100
print(success_rate)

### ligand_num_interactions and ligand size

In [ ]:
# Quick logistic curve
sns.regplot(x='ligand_num_interacting_residues', y='success', data=best_rmsd_df,
            logistic=True, ci=None); plt.axvline(8, ls='--')

In [ ]:
# Mixed-effects with size & polarity
m = smf.mixedlm(
    'rmsd ~ ligand_num_interacting_residues + ligand_molecular_weight + ligand_num_hbd + ligand_num_hba + C(method)',
    groups=best_rmsd_df['protein'], data=best_rmsd_df, family=sm.families.Binomial()
).fit()

### MW * rotatable bonds 

### MW * rotatable bonds interactino study 

In [ ]:
# ──  PREP  ──────────────────────────────────────────────────────────────
df = df_combined.copy()

# best pose per protein–method pair, then “success” flag
df = (
    df.sort_values('rmsd')
      .groupby(['protein', 'method'])
      .first()
      .reset_index()
)
thr = 2.0           # success threshold
df['success'] = (df['rmsd'] <= thr).astype(int)

# make sure you’ve columns 'ligand_molecular_weight' & 'num_rotatable_bonds'
df['mw']  = df['ligand_molecular_weight']
df['rb']  = df['ligand_num_rot_bonds']

In [ ]:
# Define “heavy” as MW > 500 Da (change if you like)
# Define “rigid” as RB <= 6    (tune after inspecting histogram)
df['weight_cat'] = np.where(df['mw'] > 500, 'heavy', 'normal')
df['flex_cat']   = np.where(df['rb'] > 6, 'floppy', 'rigid')

# Cross-tab success rates
pivot = (
    df.pivot_table(
        values='success',
        index='weight_cat',
        columns='flex_cat',
        aggfunc='mean'
    ) * 100
)
print(pivot)   # success % table

In [ ]:
# Quantile bins work well → 4×4 grid
df['mw_bin'] = pd.qcut(df['mw'], 4, labels=['Q1','Q2','Q3','Q4'])  # Q4 = heaviest
df['rb_bin'] = pd.qcut(df['rb'], 4, labels=['Q1','Q2','Q3','Q4'])  # Q4 = floppiest

heat = (
    df.groupby(['mw_bin','rb_bin'])['success']
      .mean()
      .unstack()
      .sort_index(ascending=False)   # heavies at top
)

import seaborn as sns, matplotlib.pyplot as plt
sns.heatmap(heat*100, annot=True, fmt='.0f', cmap='viridis')
plt.title('Success % vs MW quartile × RB quartile')
plt.ylabel('MW (light → heavy)'); plt.xlabel('Rotatable Bonds (rigid → floppy)')
plt.show()

“Heavy and floppy ligands are the worst.”

In [ ]:
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
# Prepare centred covariates
for col in ['mw','rb','clogP','hba','hbd']:
    df[col+'_c'] = (df[col]-df[col].mean())/df[col].std()

formula = ('success ~ mw_c*rb_c + clogP_c + hba_c + hbd_c'
           '          + C(method)')

df['protein'] = df['protein'].astype('category')
vc = {'protein': '0 + C(protein)'}   # random intercept
# vc = {'protein': '1'}
model = BinomialBayesMixedGLM.from_formula(formula, vc, df)
fit   = model.fit_vb()               # or .fit_map()

print(fit.summary())

#### Controlling for Polarity (logP/HBA/HBD)

In [ ]:
df['clogP'] = df['ligand_crippen_clogp']
df['hbd']   = df['ligand_num_hbd']
df['hba']   = df['ligand_num_hba']

# Optionally standardise
for col in ['clogP','hbd','hba']:
    df[col+'_c'] = (df[col] - df[col].mean())/df[col].std()

In [ ]:
formula = (
    'success ~ mw_c + clogP_c + hbd_c + hba_c + '
    'mw_c:rb_c + C(method)'
)
model_poly = smf.mixedlm(
    formula,
    data=df, groups=df['protein'],
    family=sm.families.Binomial()
).fit()
print(model_poly.summary())